In [2]:
import os
import pandas as pd

In [3]:
class DataLoader:

    def __init__(self, symbols_file, data_folder):

        self.symbols_file = symbols_file
        self.data_folder = data_folder

        self.columns = [
            "Symbol",
            "Date",
            "Time",
            "Open",
            "High",
            "Low",
            "Close",
            "Volume"
        ]

    # =====================================================
    # Read Symbols
    # =====================================================

    def load_symbols(self):

        if not os.path.exists(self.symbols_file):
            raise FileNotFoundError(
                f"Symbols file not found:\n{self.symbols_file}"
            )

        with open(self.symbols_file, "r") as f:

            symbols = [
                line.strip().upper()
                for line in f
                if line.strip()
            ]

        print("=" * 70)
        print(f"Total Shares : {len(symbols)}")
        print("=" * 70)

        return symbols

    # =====================================================
    # Read One Daily File
    # =====================================================

    def load_daily(self, symbol):

        filename = os.path.join(
            self.data_folder,
            f"{symbol}_weekly.txt"
        )

        if not os.path.exists(filename):

            raise FileNotFoundError(
                f"{filename} not found"
            )

        df = pd.read_csv(
            filename,
            names=self.columns,
            header=None,
            skiprows=1
        )

        # Remove blank rows
        df.dropna(inplace=True)

        # DateTime
        df["DateTime"] = pd.to_datetime(
            df["Date"] + " " + df["Time"]
        )

        # Numeric Conversion
        numeric_cols = [
            "Open",
            "High",
            "Low",
            "Close",
            "Volume"
        ]

        for col in numeric_cols:

            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

        # Remove invalid rows
        df.dropna(inplace=True)

        # Sort
        df.sort_values(
            "DateTime",
            inplace=True
        )

        # Remove duplicate candles
        df.drop_duplicates(
            subset=["DateTime"],
            inplace=True
        )

        df.reset_index(
            drop=True,
            inplace=True
        )

        return df

    # =====================================================
    # Load ALL Shares
    # =====================================================

    def load_all_daily(self):

        symbols = self.load_symbols()

        all_data = {}

        #print("\nLoading Daily Files...\n")

        for symbol in symbols:

            try:

                df = self.load_daily(symbol)

                all_data[symbol] = df

                latest = df.iloc[-1]

                #print(
                #    f"{symbol:<15}"
                 #   f"Rows : {len(df):<6}"
                 #   f"Latest : {latest['Date']}"
                #)

            except Exception as e:

                print(
                    f"{symbol:<15}FAILED : {e}"
                )

        #print()

        #print("=" * 70)
        #print(
        #    f"Successfully Loaded : {len(all_data)} Shares"
        #)
        #print("=" * 70)

        return all_data

In [4]:
SYMBOLS_FILE = "/home/hadoop/shares.txt"

DATA_FOLDER = "/home/hadoop/shareMarket_Data/weekly"

In [5]:
loader = DataLoader(

    symbols_file=SYMBOLS_FILE,

    data_folder=DATA_FOLDER

)

all_daily_data = loader.load_all_daily()

Total Shares : 80


In [6]:
from config import *

In [7]:
from IndicatorEngine import *
indicator_engine = IndicatorEngine()

Indicator Engine Initialized


In [8]:
weekly_df = all_daily_data["KALYANKJIL"]

In [9]:
weekly_df = indicator_engine.calculate(weekly_df)

In [10]:
#from IndicatorEngine import *

# Create object
#indicator_engine = IndicatorEngine()

# Calculate indicators
#daily_df = indicator_engine.calculate(daily_df)

# View last 10 rows
#daily_df.tail(10)

In [11]:
#daily_df[["Date", "+DI", "-DI", "ADX"]].tail(10)

In [12]:
#daily_df[["Date","ATR","ATR_MA"]].tail(10)

In [13]:


# ==========================================================
# SIGNAL THRESHOLDS
# ==========================================================

STRONG_BUY_SCORE = 80
BUY_SCORE = 60
WATCH_SCORE = 40

SELL_SCORE = -40
STRONG_SELL_SCORE = -80



In [15]:
from config import *
from TrendEngine import *
# Create object
trend_engine = TrendEngine()

# Run Trend Detection
trend_result = trend_engine.detect(weekly_df)

# Display Result
trend_result

Trend Engine Initialized


{'Trend': 'Bullish',
 'Signal': 'BUY',
 'Score': 78,
 'Confidence': 100.0,
 'Allowed_Trades': 'LONG ONLY',
 'Bullish': 7,
 'Bearish': 0,
 'Reasons': ['EMA20 > EMA50 > EMA200',
  'Close Above EMA20',
  'RSI Strong',
  'Positive MACD Histogram',
  'MACD Momentum Increasing',
  'Trending Market',
  '+DI Above -DI',
  'High Volume']}

In [23]:
from pprint import pprint

trend_engine = TrendEngine()

trend_result = trend_engine.detect(weekly_df)

print("=" * 60)
print("TREND ENGINE RESULT")
print("=" * 60)

pprint(trend_result)

Trend Engine Initialized
TREND ENGINE RESULT
{'Allowed_Trades': 'LONG ONLY',
 'Bearish': 0,
 'Bullish': 7,
 'Confidence': 100.0,
 'Reasons': ['EMA20 > EMA50 > EMA200',
             'Close Above EMA20',
             'RSI Strong',
             'Positive MACD Histogram',
             'MACD Momentum Increasing',
             'Trending Market',
             '+DI Above -DI',
             'High Volume'],
 'Score': 78,
 'Signal': 'BUY',
 'Trend': 'Bullish'}


In [24]:
from config import *
from SupportResistanceEngine import *

In [25]:
from config import *
from PriceActionPatternEngine import *

In [26]:
# ==========================================
# Initialize All Engines
# ==========================================

indicator_engine = IndicatorEngine()

trend_engine = TrendEngine()

sr_engine = SupportResistanceEngine()

pattern_engine = PriceActionPatternEngine()

print("All Engines Initialized Successfully.")

Indicator Engine Initialized
Trend Engine Initialized
Support & Resistance Engine Initialized
Price Action Pattern Engine initialize
All Engines Initialized Successfully.


In [27]:
# ==========================================================
# Test Price Action Pattern Engine
# ==========================================================

from pprint import pprint
import pandas as pd
import numpy as np


# ----------------------------------------------------------
# Get KALYANKJIL daily data
# ----------------------------------------------------------

daily_df = all_daily_data["KALYANKJIL"].copy()


print("=" * 80)
print("PRICE ACTION PATTERN ENGINE")
print("SYMBOL : KALYANKJIL")
print("=" * 80)


print("Rows :", len(daily_df))
print("Columns :", list(daily_df.columns))


# ==========================================================
# Prepare Required Columns
# ==========================================================

# ----------------------------------------------------------
# Make sure Volume is numeric
# ----------------------------------------------------------

daily_df["Volume"] = pd.to_numeric(
    daily_df["Volume"],
    errors="coerce"
)


# ----------------------------------------------------------
# Volume Moving Average
# ----------------------------------------------------------

if "VOLUME_MA" not in daily_df.columns:

    daily_df["VOLUME_MA"] = (
        daily_df["Volume"]
        .rolling(
            window=20,
            min_periods=1
        )
        .mean()
    )


# ----------------------------------------------------------
# Relative Volume
# ----------------------------------------------------------

if "RVOL" not in daily_df.columns:

    daily_df["RVOL"] = (
        daily_df["Volume"]
        /
        daily_df["VOLUME_MA"]
    )


# ----------------------------------------------------------
# ATR
# ----------------------------------------------------------

if "ATR" not in daily_df.columns:

    high_low = (
        daily_df["High"]
        -
        daily_df["Low"]
    )

    high_close = (
        daily_df["High"]
        -
        daily_df["Close"].shift(1)
    ).abs()

    low_close = (
        daily_df["Low"]
        -
        daily_df["Close"].shift(1)
    ).abs()

    true_range = pd.concat(
        [
            high_low,
            high_close,
            low_close
        ],
        axis=1
    ).max(axis=1)

    daily_df["ATR"] = (
        true_range
        .rolling(
            window=14,
            min_periods=1
        )
        .mean()
    )


# ==========================================================
# Check Required Columns
# ==========================================================

print("\nRequired columns:")

for column in [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "VOLUME_MA",
    "RVOL",
    "ATR"
]:

    print(
        f"{column:15} :",
        column in weekly_df.columns
    )


# ==========================================================
# Create Engine
# ==========================================================

engine = PriceActionPatternEngine()


# ==========================================================
# Detect All Patterns
# ==========================================================

result = engine.detect(
    daily_df
)


# ==========================================================
# Print Complete Result
# ==========================================================

print("\n" + "=" * 80)
print("FINAL PRICE ACTION RESULT")
print("=" * 80)

pprint(
    result,
    sort_dicts=False
)

PRICE ACTION PATTERN ENGINE
SYMBOL : KALYANKJIL
Rows : 239
Columns : ['Symbol', 'Date', 'Time', 'Open', 'High', 'Low', 'Close', 'Volume', 'DateTime']

Required columns:
Open            : True
High            : True
Low             : True
Close           : True
Volume          : True
VOLUME_MA       : True
RVOL            : True
ATR             : True
Price Action Pattern Engine initialize

FINAL PRICE ACTION RESULT
{'Bullish': False,
 'Bearish': False,
 'Market Bias': 'CONFLICTING',
 'Signal': 'HOLD',
 'Pattern': 'Falling Wedge',
 'Pattern Score': 50.0,
 'Confidence': 100.0,
 'Reasons': ['Lower High',
             'Lower Low',
             'Both trendlines falling',
             'Resistance steeper than support',
             'Converging trendlines',
             'Upside breakout',
             'High breakout volume',
             'Bullish and bearish patterns detected',
             'Strongest pattern: Falling Wedge'],
 'Detected Patterns': ['Falling Wedge', 'Lower High Lower Low'],
 

In [28]:
# ==========================================================
# Module 6 : Score Engine V2
# ==========================================================

class ScoreEngine:

    # ======================================================
    # Constructor
    # ======================================================

    def __init__(
        self,

        trend_weight=TREND_WEIGHT,

        support_resistance_weight=SUPPORT_RESISTANCE_WEIGHT,

        price_action_weight=PRICE_ACTION_WEIGHT,

        indicator_weight=INDICATOR_WEIGHT,

        buy_threshold=BUY_THRESHOLD,

        sell_threshold=SELL_THRESHOLD
    ):

        # ==================================================
        # Module Weights
        # ==================================================

        self.trend_weight = float(
            trend_weight
        )

        self.support_resistance_weight = float(
            support_resistance_weight
        )

        self.price_action_weight = float(
            price_action_weight
        )

        self.indicator_weight = float(
            indicator_weight
        )

        # ==================================================
        # Thresholds
        # ==================================================

        self.buy_threshold = float(
            buy_threshold
        )

        self.sell_threshold = float(
            sell_threshold
        )

        # ==================================================
        # Validate Weights
        # ==================================================

        total_weight = (

            self.trend_weight

            + self.support_resistance_weight

            + self.price_action_weight

            + self.indicator_weight

        )

        if abs(total_weight - 100) > 0.001:

            raise ValueError(

                "Score Engine weights must total 100."

                f"\nCurrent Total = {total_weight}"

            )

        # ==================================================
        # Print Initialization
        # ==================================================

        print(
            "Score Engine V2 Initialized"
        )


    # ======================================================
    # Utility
    # Clamp Value
    # ======================================================

    @staticmethod
    def clamp(
        value,
        minimum,
        maximum
    ):

        try:

            value = float(value)

        except (
            TypeError,
            ValueError
        ):

            value = 0.0

        return max(
            minimum,
            min(value, maximum)
        )


    # ======================================================
    # Utility
    # Safe Float
    # ======================================================

    @staticmethod
    def safe_float(
        value,
        default=0.0
    ):

        try:

            value = float(value)

            if pd.isna(value):

                return default

            return value

        except (
            TypeError,
            ValueError
        ):

            return default


    # ======================================================
    # Validate Inputs
    # ======================================================

    def validate_inputs(
        self,
        df=None,
        sr_result=None,
        trend_result=None,
        indicator_result=None,
        pattern_result=None
    ):

        if df is None:

            raise ValueError(
                "df cannot be None."
            )

        if len(df) == 0:

            raise ValueError(
                "DataFrame is empty."
            )

        if sr_result is not None:

            if not isinstance(
                sr_result,
                dict
            ):

                raise ValueError(
                    "Invalid Support/Resistance result."
                )

        if trend_result is not None:

            if not isinstance(
                trend_result,
                dict
            ):

                raise ValueError(
                    "Invalid Trend result."
                )

        if indicator_result is not None:

            if not isinstance(
                indicator_result,
                dict
            ):

                raise ValueError(
                    "Invalid Indicator result."
                )

        if pattern_result is not None:

            if not isinstance(
                pattern_result,
                dict
            ):

                raise ValueError(
                    "Invalid Pattern result."
                )

        return True


    # ======================================================
    # Create Standard Score Result
    # ======================================================

    def create_score_result(

        self,

        score=0,

        confidence=0,

        signal="HOLD",

        bullish=False,

        bearish=False,

        reasons=None,

        details=None

    ):

        if reasons is None:

            reasons = []

        if details is None:

            details = {}

        return {

            "Score": round(
                float(score),
                2
            ),

            "Confidence": round(
                float(confidence),
                2
            ),

            "Signal": signal,

            "Bullish": bool(
                bullish
            ),

            "Bearish": bool(
                bearish
            ),

            "Reasons": list(
                dict.fromkeys(
                    reasons
                )
            ),

            "Details": details

        }


    # ==========================================================
    # Part 6B
    # Calculate Indicator Score V2
    # ==========================================================

    def calculate_indicator_score(
        self,
        df
    ):

        self.validate_inputs(
            df=df
        )

        latest = df.iloc[-1]

        score = 0.0

        reasons = []

        bullish_points = 0

        bearish_points = 0

        # ==================================================
        # EMA Alignment
        # ==================================================

        ema20 = self.safe_float(
            latest.get("EMA20")
        )

        ema50 = self.safe_float(
            latest.get("EMA50")
        )

        ema200 = self.safe_float(
            latest.get("EMA200")
        )

        if (
            ema20 > 0
            and ema50 > 0
            and ema200 > 0
        ):

            if (
                ema20 >
                ema50 >
                ema200
            ):

                score += 25

                bullish_points += 1

                reasons.append(
                    "EMA20 > EMA50 > EMA200"
                )

            elif (
                ema20 <
                ema50 <
                ema200
            ):

                score -= 25

                bearish_points += 1

                reasons.append(
                    "EMA20 < EMA50 < EMA200"
                )

            elif ema20 > ema50:

                score += 10

                bullish_points += 1

                reasons.append(
                    "EMA20 above EMA50"
                )

            elif ema20 < ema50:

                score -= 10

                bearish_points += 1

                reasons.append(
                    "EMA20 below EMA50"
                )


        # ==================================================
        # RSI
        # ==================================================

        rsi = self.safe_float(
            latest.get("RSI"),
            50
        )

        if 50 <= rsi <= 70:

            score += 12

            bullish_points += 1

            reasons.append(
                "RSI bullish zone"
            )

        elif 30 <= rsi < 50:

            score -= 8

            bearish_points += 1

            reasons.append(
                "RSI below 50"
            )

        elif rsi > 70:

            score -= 4

            reasons.append(
                "RSI overbought"
            )

        elif rsi < 30:

            score += 5

            bullish_points += 1

            reasons.append(
                "RSI oversold"
            )


        # ==================================================
        # MACD
        # ==================================================

        macd = self.safe_float(
            latest.get("MACD")
        )

        macd_signal = self.safe_float(
            latest.get("MACD_SIGNAL")
        )

        macd_hist = self.safe_float(
            latest.get("MACD_HIST")
        )

        if macd > macd_signal:

            score += 12

            bullish_points += 1

            reasons.append(
                "MACD above signal"
            )

        elif macd < macd_signal:

            score -= 12

            bearish_points += 1

            reasons.append(
                "MACD below signal"
            )


        if macd_hist > 0:

            score += 6

            bullish_points += 1

            reasons.append(
                "Positive MACD histogram"
            )

        elif macd_hist < 0:

            score -= 6

            bearish_points += 1

            reasons.append(
                "Negative MACD histogram"
            )


        # ==================================================
        # ADX / DI
        # ==================================================

        adx = self.safe_float(
            latest.get("ADX")
        )

        plus_di = self.safe_float(
            latest.get("+DI")
        )

        minus_di = self.safe_float(
            latest.get("-DI")
        )

        if adx >= 25:

            if plus_di > minus_di:

                score += 10

                bullish_points += 1

                reasons.append(
                    "Strong bullish directional trend"
                )

            elif minus_di > plus_di:

                score -= 10

                bearish_points += 1

                reasons.append(
                    "Strong bearish directional trend"
                )

        else:

            reasons.append(
                "ADX indicates weak trend"
            )


        # ==================================================
        # Price vs EMA20
        # ==================================================

        close = self.safe_float(
            latest.get("Close")
        )

        if ema20 > 0:

            if close > ema20:

                score += 8

                bullish_points += 1

                reasons.append(
                    "Price above EMA20"
                )

            elif close < ema20:

                score -= 8

                bearish_points += 1

                reasons.append(
                    "Price below EMA20"
                )


        # ==================================================
        # Bollinger Bands
        # ==================================================

        bb_upper = self.safe_float(
            latest.get("BB_UPPER")
        )

        bb_lower = self.safe_float(
            latest.get("BB_LOWER")
        )

        if (
            bb_upper > 0
            and bb_lower > 0
        ):

            if close > bb_upper:

                score -= 3

                reasons.append(
                    "Price above upper Bollinger Band"
                )

            elif close < bb_lower:

                score += 3

                reasons.append(
                    "Price below lower Bollinger Band"
                )

            else:

                reasons.append(
                    "Price inside Bollinger Bands"
                )


        # ==================================================
        # Volume
        # ==================================================

        high_volume = latest.get(
            "HIGH_VOLUME",
            False
        )

        if bool(high_volume):

            if (
                bullish_points >
                bearish_points
            ):

                score += 5

                reasons.append(
                    "High volume supports bullish momentum"
                )

            elif (
                bearish_points >
                bullish_points
            ):

                score -= 5

                reasons.append(
                    "High volume supports bearish momentum"
                )


        # ==================================================
        # ATR
        # ==================================================

        atr = self.safe_float(
            latest.get("ATR")
        )

        atr_ma = self.safe_float(
            latest.get("ATR_MA")
        )

        if (
            atr > 0
            and atr_ma > 0
        ):

            if atr > atr_ma:

                reasons.append(
                    "ATR above average"
                )

            else:

                reasons.append(
                    "ATR below average"
                )


        # ==================================================
        # Normalize Score
        # ==================================================

        score = self.clamp(
            score,
            -100,
            100
        )

        # ==================================================
        # Direction
        # ==================================================

        bullish = score > 5

        bearish = score < -5

        if bullish and not bearish:

            signal = "BUY"

        elif bearish and not bullish:

            signal = "SELL"

        else:

            signal = "HOLD"


        # ==================================================
        # Indicator Confidence
        # ==================================================

        total_directional_votes = (
            bullish_points
            + bearish_points
        )

        if total_directional_votes > 0:

            directional_agreement = (

                max(
                    bullish_points,
                    bearish_points
                )

                / total_directional_votes

            ) * 100

        else:

            directional_agreement = 0


        confidence = (

            abs(score) * 0.70

            + directional_agreement * 0.30

        )

        confidence = self.clamp(
            confidence,
            0,
            100
        )


        # ==================================================
        # Details
        # ==================================================

        details = {

            "Bullish Votes":
                bullish_points,

            "Bearish Votes":
                bearish_points,

            "Directional Agreement":
                round(
                    directional_agreement,
                    2
                )

        }


        return self.create_score_result(

            score=score,

            confidence=confidence,

            signal=signal,

            bullish=bullish,

            bearish=bearish,

            reasons=reasons,

            details=details

        )


    # ==========================================================
    # Part 6C
    # Calculate Trend Score V2
    # ==========================================================

    def calculate_trend_score(
        self,
        trend_result
    ):

        if trend_result is None:

            raise ValueError(
                "Trend result cannot be None."
            )

        raw_score = self.safe_float(
            trend_result.get(
                "Score",
                0
            )
        )

        trend = str(
            trend_result.get(
                "Trend",
                ""
            )
        ).strip().lower()

        signal = str(
            trend_result.get(
                "Signal",
                ""
            )
        ).strip().upper()

        confidence = self.clamp(
            self.safe_float(
                trend_result.get(
                    "Confidence",
                    0
                )
            ),
            0,
            100
        )

        reasons = list(
            trend_result.get(
                "Reasons",
                []
            )
        )

        # ==================================================
        # Normalize Existing Trend Score
        # ==================================================

        score = self.clamp(
            raw_score,
            -100,
            100
        )

        # ==================================================
        # Trend Direction
        # ==================================================

        if trend == "bullish":

            score = max(
                score,
                50
            )

            bullish = True

            bearish = False

            reasons.append(
                "Bullish Trend"
            )

        elif trend == "bearish":

            score = min(
                score,
                -50
            )

            bullish = False

            bearish = True

            reasons.append(
                "Bearish Trend"
            )

        else:

            bullish = False

            bearish = False

            reasons.append(
                "Sideways / Neutral Trend"
            )


        # ==================================================
        # Signal Agreement
        # ==================================================

        if (
            signal == "BUY"
            and bullish
        ):

            score = min(
                score + 10,
                100
            )

            reasons.append(
                "Trend BUY signal confirmed"
            )

        elif (
            signal == "SELL"
            and bearish
        ):

            score = max(
                score - 10,
                -100
            )

            reasons.append(
                "Trend SELL signal confirmed"
            )


        # ==================================================
        # Final Signal
        # ==================================================

        if bullish and not bearish:

            final_signal = "BUY"

        elif bearish and not bullish:

            final_signal = "SELL"

        else:

            final_signal = "HOLD"


        return self.create_score_result(

            score=score,

            confidence=confidence,

            signal=final_signal,

            bullish=bullish,

            bearish=bearish,

            reasons=reasons

        )


    # ==========================================================
    # Part 6D
    # Calculate Support / Resistance Score V2
    # ==========================================================

    def calculate_support_resistance_score(
        self,
        df,
        sr_result
    ):

        self.validate_inputs(

            df=df,

            sr_result=sr_result

        )

        latest = df.iloc[-1]

        close = self.safe_float(
            latest.get("Close")
        )

        score = 0.0

        bullish = False

        bearish = False

        reasons = []

        details = {}


        # ==================================================
        # Major Support
        # ==================================================

        support = sr_result.get(
            "major_support"
        )

        if support is not None:

            support_price = self.safe_float(
                support.get("price")
            )

            zone_low = self.safe_float(
                support.get("zone_low")
            )

            zone_high = self.safe_float(
                support.get("zone_high")
            )

            details[
                "Major Support"
            ] = support_price


            if (
                zone_low <= close <= zone_high
            ):

                score += 30

                bullish = True

                reasons.append(
                    "Price inside Major Support Zone"
                )

            elif close > zone_high:

                if support_price > 0:

                    distance = (

                        (close - support_price)

                        / support_price

                    ) * 100

                else:

                    distance = 100


                details[
                    "Support Distance %"
                ] = round(
                    distance,
                    2
                )


                if distance <= 2:

                    score += 25

                    bullish = True

                    reasons.append(
                        "Price near Major Support"
                    )

                elif distance <= 5:

                    score += 15

                    bullish = True

                    reasons.append(
                        "Price above Support"
                    )

                elif distance <= 10:

                    score += 5

                    bullish = True

                    reasons.append(
                        "Support below price"
                    )

            else:

                score -= 30

                bearish = True

                reasons.append(
                    "Price below Major Support"
                )


        # ==================================================
        # Major Resistance
        # ==================================================

        resistance = sr_result.get(
            "major_resistance"
        )

        if resistance is not None:

            resistance_price = self.safe_float(
                resistance.get("price")
            )

            zone_low = self.safe_float(
                resistance.get("zone_low")
            )

            zone_high = self.safe_float(
                resistance.get("zone_high")
            )

            details[
                "Major Resistance"
            ] = resistance_price


            if (
                zone_low <= close <= zone_high
            ):

                score -= 30

                bearish = True

                reasons.append(
                    "Price inside Major Resistance Zone"
                )

            elif close < zone_low:

                if resistance_price > 0:

                    distance = (

                        (resistance_price - close)

                        / resistance_price

                    ) * 100

                else:

                    distance = 100


                details[
                    "Resistance Distance %"
                ] = round(
                    distance,
                    2
                )


                if distance <= 2:

                    score -= 25

                    bearish = True

                    reasons.append(
                        "Price near Major Resistance"
                    )

                elif distance <= 5:

                    score -= 15

                    bearish = True

                    reasons.append(
                        "Price below Resistance"
                    )

                elif distance <= 10:

                    score -= 5

                    bearish = True

                    reasons.append(
                        "Resistance above price"
                    )

            else:

                score += 30

                bullish = True

                reasons.append(
                    "Price broke above Major Resistance"
                )


        # ==================================================
        # Pivot
        # ==================================================

        pivot = sr_result.get(
            "pivot"
        )

        if pivot:

            pp = self.safe_float(
                pivot.get("PP")
            )

            details[
                "Pivot"
            ] = pp

            buffer = pp * 0.005


            if close > pp + buffer:

                score += 10

                bullish = True

                reasons.append(
                    "Price above Pivot"
                )

            elif close < pp - buffer:

                score -= 10

                bearish = True

                reasons.append(
                    "Price below Pivot"
                )

            else:

                reasons.append(
                    "Price near Pivot"
                )


        # ==================================================
        # Clamp
        # ==================================================

        score = self.clamp(
            score,
            -100,
            100
        )


        # ==================================================
        # Signal
        # ==================================================

        if bullish and not bearish:

            signal = "BUY"

        elif bearish and not bullish:

            signal = "SELL"

        else:

            signal = "HOLD"


        # ==================================================
        # Confidence
        # ==================================================

        confidence = min(
            abs(score),
            100
        )


        return self.create_score_result(

            score=score,

            confidence=confidence,

            signal=signal,

            bullish=bullish,

            bearish=bearish,

            reasons=reasons,

            details=details

        )


    # ==========================================================
    # Part 6E
    # Calculate Price Action Pattern Score V2
    # ==========================================================

    def calculate_pattern_score(
        self,
        pattern_result
    ):

        if pattern_result is None:

            raise ValueError(
                "Pattern result cannot be None."
            )


        detected = pattern_result.get(
            "Detected Patterns",
            []
        )

        if not detected:

            return self.create_score_result(

                score=0,

                confidence=0,

                signal="HOLD",

                reasons=[
                    "No Price Action Pattern"
                ],

                details={}

            )


        # ==================================================
        # Strongest Pattern
        # ==================================================

        pattern = pattern_result.get(
            "Pattern",
            "None"
        )

        pattern_score = self.clamp(

            self.safe_float(
                pattern_result.get(
                    "Pattern Score",
                    0
                )
            ),

            0,

            100

        )

        pattern_confidence = self.clamp(

            self.safe_float(
                pattern_result.get(
                    "Confidence",
                    0
                )
            ),

            0,

            100

        )


        bullish_patterns = pattern_result.get(
            "Bullish Patterns",
            []
        )

        bearish_patterns = pattern_result.get(
            "Bearish Patterns",
            []
        )


        bullish = len(
            bullish_patterns
        ) > 0

        bearish = len(
            bearish_patterns
        ) > 0


        reasons = list(
            pattern_result.get(
                "Reasons",
                []
            )
        )


        details = {

            "Best Pattern":
                pattern,

            "Detected Patterns":
                detected,

            "Bullish Patterns":
                bullish_patterns,

            "Bearish Patterns":
                bearish_patterns,

            "Raw Pattern Score":
                pattern_score,

            "Pattern Confidence":
                pattern_confidence

        }


        # ==================================================
        # Direction of Strongest Pattern
        # ==================================================

        best_bullish = (
            pattern_result.get(
                "Best Pattern",
                {}
            ).get(
                "Bullish",
                False
            )
        )

        best_bearish = (
            pattern_result.get(
                "Best Pattern",
                {}
            ).get(
                "Bearish",
                False
            )
        )


        # ==================================================
        # Convert 0-100 Pattern Score
        # to -100/+100 Directional Score
        # ==================================================

        if (
            best_bullish
            and not best_bearish
        ):

            score = pattern_score

            signal = "BUY"

        elif (
            best_bearish
            and not best_bullish
        ):

            score = -pattern_score

            signal = "SELL"

        else:

            # ---------------------------------------------
            # If master pattern result is conflicting,
            # calculate directional balance.
            # ---------------------------------------------

            bullish_strength = 0

            bearish_strength = 0


            if bullish_patterns:

                bullish_strength = pattern_score


            if bearish_patterns:

                bearish_strength = pattern_score


            if (
                bullish_strength >
                bearish_strength
            ):

                score = bullish_strength

                signal = "BUY"

            elif (
                bearish_strength >
                bullish_strength
            ):

                score = -bearish_strength

                signal = "SELL"

            else:

                score = 0

                signal = "HOLD"


        # ==================================================
        # Conflict Handling
        # ==================================================

        conflict = (

            bullish
            and bearish

        )


        if conflict:

            score *= 0.70

            confidence = (
                pattern_confidence
                * 0.70
            )

            reasons.append(
                "Bullish and bearish patterns conflict"
            )

            details[
                "Conflict"
            ] = True

        else:

            confidence = (
                pattern_confidence
            )

            details[
                "Conflict"
            ] = False


        # ==================================================
        # Final Direction
        # ==================================================

        bullish = score > 5

        bearish = score < -5


        if bullish and not bearish:

            signal = "BUY"

        elif bearish and not bullish:

            signal = "SELL"

        else:

            signal = "HOLD"


        # ==================================================
        # Store Signed Score
        # ==================================================

        details[
            "Signed Pattern Score"
        ] = round(
            score,
            2
        )


        return self.create_score_result(

            score=score,

            confidence=confidence,

            signal=signal,

            bullish=bullish,

            bearish=bearish,

            reasons=reasons,

            details=details

        )


    # ==========================================================
    # Part 6F
    # Calculate Final Weighted Score V2
    # ==========================================================

    def calculate_total_score(

        self,

        indicator_score,

        trend_score,

        sr_score,

        pattern_score

    ):

        if indicator_score is None:

            raise ValueError(
                "Indicator score is missing."
            )

        if trend_score is None:

            raise ValueError(
                "Trend score is missing."
            )

        if sr_score is None:

            raise ValueError(
                "Support/Resistance score is missing."
            )

        if pattern_score is None:

            raise ValueError(
                "Pattern score is missing."
            )


        # ==================================================
        # Weighted Module Scores
        # ==================================================

        indicator_weighted = (

            indicator_score["Score"]

            * self.indicator_weight

            / 100

        )


        trend_weighted = (

            trend_score["Score"]

            * self.trend_weight

            / 100

        )


        sr_weighted = (

            sr_score["Score"]

            * self.support_resistance_weight

            / 100

        )


        pattern_weighted = (

            pattern_score["Score"]

            * self.price_action_weight

            / 100

        )


        # ==================================================
        # Final Score
        # Range: -100 to +100
        # ==================================================

        total_score = (

            indicator_weighted

            + trend_weighted

            + sr_weighted

            + pattern_weighted

        )


        total_score = self.clamp(

            total_score,

            -100,

            100

        )


        # ==================================================
        # Weighted Confidence
        # ==================================================

        confidence = (

            indicator_score["Confidence"]
            * self.indicator_weight

            +

            trend_score["Confidence"]
            * self.trend_weight

            +

            sr_score["Confidence"]
            * self.support_resistance_weight

            +

            pattern_score["Confidence"]
            * self.price_action_weight

        ) / 100


        confidence = self.clamp(

            confidence,

            0,

            100

        )


        # ==================================================
        # Directional Votes
        # ==================================================

        bullish_votes = sum([

            bool(
                indicator_score["Bullish"]
            ),

            bool(
                trend_score["Bullish"]
            ),

            bool(
                sr_score["Bullish"]
            ),

            bool(
                pattern_score["Bullish"]
            )

        ])


        bearish_votes = sum([

            bool(
                indicator_score["Bearish"]
            ),

            bool(
                trend_score["Bearish"]
            ),

            bool(
                sr_score["Bearish"]
            ),

            bool(
                pattern_score["Bearish"]
            )

        ])


        # ==================================================
        # Conflict Detection
        # ==================================================

        directional_conflict = (

            bullish_votes > 0

            and

            bearish_votes > 0

        )


        # ==================================================
        # Final Signal
        # ==================================================

        if (

            total_score >= self.buy_threshold

            and bullish_votes >= 3

            and not directional_conflict

        ):

            signal = "BUY"


        elif (

            total_score <= self.sell_threshold

            and bearish_votes >= 3

            and not directional_conflict

        ):

            signal = "SELL"


        else:

            signal = "HOLD"


        # ==================================================
        # Final Direction
        # ==================================================

        if signal == "BUY":

            bullish = True

            bearish = False

        elif signal == "SELL":

            bullish = False

            bearish = True

        else:

            bullish = False

            bearish = False


        # ==================================================
        # Reasons
        # ==================================================

        reasons = []

        reasons.extend(
            indicator_score["Reasons"]
        )

        reasons.extend(
            trend_score["Reasons"]
        )

        reasons.extend(
            sr_score["Reasons"]
        )

        reasons.extend(
            pattern_score["Reasons"]
        )


        reasons = list(
            dict.fromkeys(
                reasons
            )
        )


        if directional_conflict:

            reasons.append(
                "Directional conflict between scoring modules"
            )


        if signal == "BUY":

            reasons.append(
                "Final BUY conditions satisfied"
            )

        elif signal == "SELL":

            reasons.append(
                "Final SELL conditions satisfied"
            )

        else:

            reasons.append(
                "Final BUY/SELL conditions not satisfied"
            )


        # ==================================================
        # Details
        # ==================================================

        details = {

            "Indicator Score":
                round(
                    indicator_score["Score"],
                    2
                ),

            "Trend Score":
                round(
                    trend_score["Score"],
                    2
                ),

            "Support/Resistance Score":
                round(
                    sr_score["Score"],
                    2
                ),

            "Pattern Score":
                round(
                    pattern_score["Score"],
                    2
                ),

            "Weighted Indicator":
                round(
                    indicator_weighted,
                    2
                ),

            "Weighted Trend":
                round(
                    trend_weighted,
                    2
                ),

            "Weighted Support":
                round(
                    sr_weighted,
                    2
                ),

            "Weighted Pattern":
                round(
                    pattern_weighted,
                    2
                ),

            "Bullish Votes":
                bullish_votes,

            "Bearish Votes":
                bearish_votes,

            "Directional Conflict":
                directional_conflict,

            "Buy Threshold":
                self.buy_threshold,

            "Sell Threshold":
                self.sell_threshold

        }


        return self.create_score_result(

            score=total_score,

            confidence=confidence,

            signal=signal,

            bullish=bullish,

            bearish=bearish,

            reasons=reasons,

            details=details

        )


    # ==========================================================
    # Part 6G
    # Main Score Engine
    # ==========================================================

    def calculate(

        self,

        df,

        trend_result,

        sr_result,

        pattern_result=None

    ):

        """
        Main Score Engine V2.

        Pipeline:

        Data
          ↓
        Indicator Score
          ↓
        Trend Score
          ↓
        Support / Resistance Score
          ↓
        Price Action Pattern Score
          ↓
        Weighted Final Score
          ↓
        BUY / SELL / HOLD
        """

        # ==================================================
        # Validation
        # ==================================================

        self.validate_inputs(

            df=df,

            trend_result=trend_result,

            sr_result=sr_result,

            pattern_result=pattern_result

        )


        # ==================================================
        # Indicator
        # ==================================================

        indicator_score = (

            self.calculate_indicator_score(
                df
            )

        )


        # ==================================================
        # Trend
        # ==================================================

        trend_score = (

            self.calculate_trend_score(
                trend_result
            )

        )


        # ==================================================
        # Support / Resistance
        # ==================================================

        sr_score = (

            self.calculate_support_resistance_score(

                df,

                sr_result

            )

        )


        # ==================================================
        # Price Action Pattern
        # ==================================================

        if pattern_result is None:

            pattern_score = (

                self.create_score_result(

                    score=0,

                    confidence=0,

                    signal="HOLD",

                    bullish=False,

                    bearish=False,

                    reasons=[
                        "Pattern module not available"
                    ]

                )

            )

        else:

            pattern_score = (

                self.calculate_pattern_score(

                    pattern_result

                )

            )


        # ==================================================
        # Final Weighted Score
        # ==================================================

        total_score = (

            self.calculate_total_score(

                indicator_score,

                trend_score,

                sr_score,

                pattern_score

            )

        )


        # ==================================================
        # Attach Module Results
        # ==================================================

        total_score[
            "Indicator Score"
        ] = indicator_score


        total_score[
            "Trend Score"
        ] = trend_score


        total_score[
            "Support Resistance Score"
        ] = sr_score


        total_score[
            "Pattern Score"
        ] = pattern_score


        # ==================================================
        # Module Weight Information
        # ==================================================

        total_score[
            "Weights"
        ] = {

            "Trend":
                self.trend_weight,

            "Support Resistance":
                self.support_resistance_weight,

            "Price Action":
                self.price_action_weight,

            "Indicator":
                self.indicator_weight

        }


        return total_score

In [29]:
# ==========================================================
# Prepare Daily Data
# ==========================================================

symbol = "KALYANKJIL"

_df = all_daily_data[symbol].copy()

print("=" * 80)
print("PREPARING DATA")
print("=" * 80)

print("Raw columns:")
print(list(daily_df.columns))


# ==========================================================
# Indicator Engine
# ==========================================================

indicator_engine = IndicatorEngine()

daily_df = indicator_engine.calculate(
    daily_df
)


# ==========================================================
# Verify Required Columns
# ==========================================================

required_columns = [

    "Open",
    "High",
    "Low",
    "Close",
    "Volume",

    "EMA20",
    "EMA50",
    "EMA200",

    "RSI",

    "MACD",
    "MACD_SIGNAL",
    "MACD_HIST",

    "ADX",
    "+DI",
    "-DI",

    "ATR",
    "ATR_MA",

    "VOLUME_MA",
    "RVOL",

    "BB_UPPER",
    "BB_LOWER"

]

print("\n" + "=" * 80)
print("INDICATOR COLUMN CHECK")
print("=" * 80)

for column in required_columns:

    print(
        f"{column:<20}:",
        column in daily_df.columns
    )

PREPARING DATA
Raw columns:
['Symbol', 'Date', 'Time', 'Open', 'High', 'Low', 'Close', 'Volume', 'DateTime', 'VOLUME_MA', 'RVOL', 'ATR']
Indicator Engine Initialized

INDICATOR COLUMN CHECK
Open                : True
High                : True
Low                 : True
Close               : True
Volume              : True
EMA20               : True
EMA50               : True
EMA200              : True
RSI                 : True
MACD                : True
MACD_SIGNAL         : True
MACD_HIST           : True
ADX                 : True
+DI                 : True
-DI                 : True
ATR                 : True
ATR_MA              : True
VOLUME_MA           : True
RVOL                : True
BB_UPPER            : True
BB_LOWER            : True


In [30]:
# ==========================================================
# Price Action Pattern Engine
# ==========================================================

pattern_engine = PriceActionPatternEngine()

pattern_result = pattern_engine.detect(
    daily_df
)

print("\n" + "=" * 80)
print("PRICE ACTION RESULT")
print("=" * 80)

from pprint import pprint

pprint(pattern_result)

Price Action Pattern Engine initialize

PRICE ACTION RESULT
{'Bearish': False,
 'Bearish Patterns': ['Lower High Lower Low'],
 'Best Pattern': {'Bars': 239,
                  'Bearish': False,
                  'Bullish': True,
                  'Confidence': 100.0,
                  'Details': {'Channel Width': np.float64(91.06),
                              'Latest Close': np.float64(606.3),
                              'Lower Slope': np.float64(-3.6517),
                              'Lower Trendline': np.float64(290.91),
                              'Slope Difference': np.float64(0.8111),
                              'Upper Slope': np.float64(-4.4628),
                              'Upper Trendline': np.float64(381.97)},
                  'Detected': True,
                  'LatestClose': 606.3,
                  'LatestDate': '2026-08-07',
                  'Pattern': 'Falling Wedge',
                  'Reasons': ['Both trendlines falling',
                              'Resis

In [31]:
# ==========================================================
# Trend Engine
# ==========================================================

trend_engine = TrendEngine()

trend_result = trend_engine.detect(
    daily_df
)

print("\n" + "=" * 80)
print("TREND RESULT")
print("=" * 80)

pprint(trend_result)

Trend Engine Initialized

TREND RESULT
{'Allowed_Trades': 'LONG ONLY',
 'Bearish': 0,
 'Bullish': 7,
 'Confidence': 100.0,
 'Reasons': ['EMA20 > EMA50 > EMA200',
             'Close Above EMA20',
             'RSI Strong',
             'Positive MACD Histogram',
             'MACD Momentum Increasing',
             'Trending Market',
             '+DI Above -DI',
             'High Volume'],
 'Score': 78,
 'Signal': 'BUY',
 'Trend': 'Bullish'}


In [32]:
# ==========================================================
# SCORE ENGINE V2 TEST
# ==========================================================

from pprint import pprint

symbol = "KALYANKJIL"

# ==========================================================
# 1. Load Raw Daily Data
# ==========================================================

daily_df = all_daily_data[symbol].copy()

print("=" * 80)
print("SCORE ENGINE V2 TEST")
print("=" * 80)

print("SYMBOL :", symbol)
print("Rows   :", len(daily_df))

print("\nRAW COLUMNS:")
print(daily_df.columns.tolist())

# ==========================================================
# 2. Calculate Indicators
# ==========================================================

print("\n" + "=" * 80)
print("CALCULATING INDICATORS")
print("=" * 80)

indicator_engine = IndicatorEngine()

daily_df = indicator_engine.calculate(
    daily_df
)

# ==========================================================
# 3. Verify Required Columns
# ==========================================================

required_columns = [

    "EMA20",
    "EMA50",
    "EMA200",

    "RSI",

    "MACD",
    "MACD_SIGNAL",
    "MACD_HIST",

    "ADX",
    "+DI",
    "-DI",

    "VOLUME_MA",
    "RVOL",

    "ATR",
    "ATR_MA"
]

print("\nINDICATOR COLUMN CHECK")
print("=" * 80)

missing_columns = []

for column in required_columns:

    exists = column in daily_df.columns

    print(
        f"{column:<20}: {exists}"
    )

    if not exists:
        missing_columns.append(column)

if missing_columns:

    raise ValueError(
        "Missing indicator columns: "
        + ", ".join(missing_columns)
    )

print("\nAll required indicator columns are available.")

SCORE ENGINE V2 TEST
SYMBOL : KALYANKJIL
Rows   : 239

RAW COLUMNS:
['Symbol', 'Date', 'Time', 'Open', 'High', 'Low', 'Close', 'Volume', 'DateTime']

CALCULATING INDICATORS
Indicator Engine Initialized

INDICATOR COLUMN CHECK
EMA20               : True
EMA50               : True
EMA200              : True
RSI                 : True
MACD                : True
MACD_SIGNAL         : True
MACD_HIST           : True
ADX                 : True
+DI                 : True
-DI                 : True
VOLUME_MA           : True
RVOL                : True
ATR                 : True
ATR_MA              : True

All required indicator columns are available.


In [33]:
# ==========================================================
# 4. Price Action Pattern Engine
# ==========================================================

pattern_engine = PriceActionPatternEngine()

pattern_result = pattern_engine.detect(
    daily_df
)

print("\n" + "=" * 80)
print("PRICE ACTION RESULT")
print("=" * 80)

print(
    "Best Pattern :",
    pattern_result.get("Pattern")
)

print(
    "Pattern Score :",
    pattern_result.get("Pattern Score")
)

print(
    "Confidence :",
    pattern_result.get("Confidence")
)

print(
    "Market Bias :",
    pattern_result.get("Market Bias")
)

print(
    "Signal :",
    pattern_result.get("Signal")
)

print(
    "Detected Patterns :",
    pattern_result.get("Detected Patterns")
)

Price Action Pattern Engine initialize

PRICE ACTION RESULT
Best Pattern : Falling Wedge
Pattern Score : 50.0
Confidence : 100.0
Market Bias : CONFLICTING
Signal : HOLD
Detected Patterns : ['Falling Wedge', 'Lower High Lower Low']


In [34]:
# ==========================================================
# 5. Trend Engine
# ==========================================================

trend_engine = TrendEngine()

trend_result = trend_engine.detect(
    daily_df
)

print("\n" + "=" * 80)
print("TREND RESULT")
print("=" * 80)

pprint(trend_result)

Trend Engine Initialized

TREND RESULT
{'Allowed_Trades': 'LONG ONLY',
 'Bearish': 0,
 'Bullish': 7,
 'Confidence': 100.0,
 'Reasons': ['EMA20 > EMA50 > EMA200',
             'Close Above EMA20',
             'RSI Strong',
             'Positive MACD Histogram',
             'MACD Momentum Increasing',
             'Trending Market',
             '+DI Above -DI',
             'High Volume'],
 'Score': 78,
 'Signal': 'BUY',
 'Trend': 'Bullish'}


In [35]:
# ==========================================================
# 5. Support / Resistance Engine
# ==========================================================

sr_engine = SupportResistanceEngine()

sr_result = sr_engine.calculate(
    daily_df
)

print("\n" + "=" * 80)
print("SUPPORT / RESISTANCE RESULT")
print("=" * 80)

pprint(sr_result)

Support & Resistance Engine Initialized

SUPPORT / RESISTANCE RESULT
{'major_resistance': {'datetime': Timestamp('2025-07-25 00:00:00'),
                      'price': np.float64(617.7),
                      'strength': 67,
                      'touches': 9,
                      'volume': np.float64(20223969.0),
                      'zone_high': np.float64(620.79),
                      'zone_low': np.float64(614.61)},
 'major_support': {'datetime': Timestamp('2025-06-27 00:00:00'),
                   'price': np.float64(502.35),
                   'strength': 70,
                   'touches': 35,
                   'volume': np.float64(28938756.0),
                   'zone_high': np.float64(504.86),
                   'zone_low': np.float64(499.84)},
 'nearest_resistance': {'datetime': Timestamp('2025-07-25 00:00:00'),
                        'index': 184,
                        'price': np.float64(617.7),
                        'strength': 67,
                        'touches':

In [36]:
# ==========================================================
# 6. Score Engine
# ==========================================================

score_engine = ScoreEngine()

final_result = score_engine.calculate(
    df=daily_df,
    trend_result=trend_result,
    sr_result=sr_result,
    pattern_result=pattern_result
)

# ==========================================================
# 7. Print Complete Final Result
# ==========================================================

from pprint import pprint

print("\n" + "=" * 80)
print("FINAL SCORE ENGINE RESULT")
print("=" * 80)

pprint(final_result)

# ==========================================================
# 8. Final Summary
# ==========================================================

print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

print("Symbol     :", symbol)
print("Signal     :", final_result.get("Signal"))
print("Score      :", final_result.get("Score"))
print("Confidence :", final_result.get("Confidence"))
print("Bullish    :", final_result.get("Bullish"))
print("Bearish    :", final_result.get("Bearish"))

# ==========================================================
# 9. Module Scores
# ==========================================================

print("\n" + "=" * 80)
print("MODULE SCORES")
print("=" * 80)

print(
    "Indicator Score :",
    final_result["Indicator Score"]["Score"]
)

print(
    "Trend Score     :",
    final_result["Trend Score"]["Score"]
)

print(
    "Support/Res Score :",
    final_result["Support Resistance Score"]["Score"]
)

print(
    "Pattern Score   :",
    final_result["Pattern Score"]["Score"]
)

# ==========================================================
# 10. Weighted Scores
# ==========================================================

print("\n" + "=" * 80)
print("WEIGHTED SCORES")
print("=" * 80)

details = final_result.get("Details", {})

print(
    "Weighted Indicator :",
    details.get("Weighted Indicator")
)

print(
    "Weighted Trend     :",
    details.get("Weighted Trend")
)

print(
    "Weighted Support   :",
    details.get("Weighted Support")
)

print(
    "Weighted Pattern   :",
    details.get("Weighted Pattern")
)

# ==========================================================
# 11. Bullish / Bearish Votes
# ==========================================================

print("\n" + "=" * 80)
print("BULLISH / BEARISH VOTES")
print("=" * 80)

print(
    "Bullish Votes :",
    details.get("Bullish Votes")
)

print(
    "Bearish Votes :",
    details.get("Bearish Votes")
)

# ==========================================================
# 12. Final Reasons
# ==========================================================

print("\n" + "=" * 80)
print("FINAL REASONS")
print("=" * 80)

for i, reason in enumerate(
    final_result.get("Reasons", []),
    start=1
):
    print(f"{i:02d}. {reason}")

Score Engine V2 Initialized

FINAL SCORE ENGINE RESULT
{'Bearish': False,
 'Bullish': False,
 'Confidence': 69.44,
 'Details': {'Bearish Votes': 1,
             'Bullish Votes': 3,
             'Buy Threshold': 70.0,
             'Directional Conflict': True,
             'Indicator Score': 59.0,
             'Pattern Score': 35.0,
             'Sell Threshold': 30.0,
             'Support/Resistance Score': -25.0,
             'Trend Score': 88.0,
             'Weighted Indicator': 8.85,
             'Weighted Pattern': 8.75,
             'Weighted Support': -6.25,
             'Weighted Trend': 30.8},
 'Indicator Score': {'Bearish': False,
                     'Bullish': True,
                     'Confidence': 71.3,
                     'Details': {'Bearish Votes': 0,
                                 'Bullish Votes': 5,
                                 'Directional Agreement': 100.0},
                     'Reasons': ['EMA20 > EMA50 > EMA200',
                                 'RSI ov

In [37]:
# ==========================================================
# Module 7 : Report Writer V3
# ==========================================================

import os
import pandas as pd


class ReportWriter:

    # ======================================================
    # Constructor
    # ======================================================

    def __init__(
        self,
        output_folder="report/daily",
        minimum_confidence=60
    ):

        # --------------------------------------------------
        # Output folder
        # --------------------------------------------------

        self.output_folder = output_folder

        # --------------------------------------------------
        # Minimum confidence for category reports
        # --------------------------------------------------

        self.minimum_confidence = minimum_confidence

        # --------------------------------------------------
        # Create directory
        # --------------------------------------------------

        os.makedirs(
            self.output_folder,
            exist_ok=True
        )

        print(
            "Report Writer V3 Initialized"
        )

        print(
            "Output Folder :",
            self.output_folder
        )

    # ======================================================
    # Safe Value
    # ======================================================

    def _safe_value(
        self,
        value,
        default=None
    ):
        """
        Safely return a value.

        Handles:
            None
            NaN
            pandas values
        """

        if value is None:
            return default

        try:

            if pd.isna(value):
                return default

        except Exception:

            pass

        return value

    # ======================================================
    # Convert List To Text
    # ======================================================

    def _list_to_text(
        self,
        value
    ):
        """
        Convert list values into readable
        Excel text.
        """

        if value is None:
            return ""

        if isinstance(
            value,
            list
        ):

            return ", ".join(
                str(item)
                for item in value
            )

        return str(value)

    # ======================================================
    # Extract Support
    # ======================================================

    def _extract_support(
        self,
        sr_result
    ):

        if not isinstance(
            sr_result,
            dict
        ):

            return {}

        support = sr_result.get(
            "major_support"
        )

        if not isinstance(
            support,
            dict
        ):

            return {}

        return support

    # ======================================================
    # Extract Resistance
    # ======================================================

    def _extract_resistance(
        self,
        sr_result
    ):

        if not isinstance(
            sr_result,
            dict
        ):

            return {}

        resistance = sr_result.get(
            "major_resistance"
        )

        if not isinstance(
            resistance,
            dict
        ):

            return {}

        return resistance

    # ======================================================
    # Get Module Score
    # ======================================================

    def _get_module_score(
        self,
        total_score,
        details_key,
        direct_key
    ):
        """
        Handles both possible ScoreEngine formats.

        Example:

        Details["Indicator Score"]

        OR

        total_score["Indicator Score"]
        """

        details = total_score.get(
            "Details",
            {}
        )

        value = details.get(
            details_key
        )

        if value is None:

            value = total_score.get(
                direct_key,
                {}
            )

        # --------------------------------------------------
        # If dictionary
        # --------------------------------------------------

        if isinstance(
            value,
            dict
        ):

            return value

        # --------------------------------------------------
        # If numeric
        # --------------------------------------------------

        return {
            "Score": value
        }

    # ======================================================
    # Remove Symbol From File
    # ======================================================

    def _remove_symbol_from_file(
        self,
        filename,
        symbol
    ):
        """
        Remove an existing symbol from an Excel file.

        This prevents a stock from remaining in an
        old BUY/SELL/HOLD file after its signal changes.
        """

        if not os.path.exists(
            filename
        ):

            return

        try:

            df = pd.read_excel(
                filename
            )

        except Exception as e:

            print(
                f"[WARNING] Could not read "
                f"{filename}: {e}"
            )

            return

        if "Symbol" not in df.columns:

            return

        df = df[
            df["Symbol"] != symbol
        ]

        df.to_excel(
            filename,
            index=False
        )

    # ======================================================
    # Upsert Excel Row
    # ======================================================

    def _upsert_row(
        self,
        filename,
        row
    ):
        """
        Add or update one symbol in an Excel file.

        Existing symbol:
            replaced

        New symbol:
            appended
        """

        if os.path.exists(
            filename
        ):

            try:

                existing_df = pd.read_excel(
                    filename
                )

            except Exception as e:

                print(
                    f"[WARNING] Could not read "
                    f"{filename}: {e}"
                )

                existing_df = pd.DataFrame()

        else:

            existing_df = pd.DataFrame()

        # --------------------------------------------------
        # Append new row
        # --------------------------------------------------

        new_df = pd.DataFrame(
            [row]
        )

        existing_df = pd.concat(
            [
                existing_df,
                new_df
            ],
            ignore_index=True
        )

        # --------------------------------------------------
        # Remove duplicate symbols
        # --------------------------------------------------

        if "Symbol" in existing_df.columns:

            existing_df = (
                existing_df
                .drop_duplicates(
                    subset=["Symbol"],
                    keep="last"
                )
            )

        # --------------------------------------------------
        # Save
        # --------------------------------------------------

        existing_df.to_excel(
            filename,
            index=False
        )

    # ======================================================
    # Get Output Filename
    # ======================================================

    def _get_category_filename(
        self,
        signal
    ):

        if signal == "BUY":

            return os.path.join(
                self.output_folder,
                "Bullish_Shares.xlsx"
            )

        elif signal == "SELL":

            return os.path.join(
                self.output_folder,
                "Bearish_Shares.xlsx"
            )

        else:

            return os.path.join(
                self.output_folder,
                "Hold_Shares.xlsx"
            )

    # ======================================================
    # Save Share
    # ======================================================

    def save_share(
        self,
        symbol,
        total_score,
        trend_result,
        sr_result,
        pattern_result,
        latest_row
    ):
        """
        Save complete analysis for one share.

        IMPORTANT:

        Daily_Analysis.xlsx
            -> ALWAYS receives the share.

        Bullish_Shares.xlsx
            -> BUY shares with sufficient confidence.

        Bearish_Shares.xlsx
            -> SELL shares with sufficient confidence.

        Hold_Shares.xlsx
            -> HOLD shares with sufficient confidence.
        """

        # ==================================================
        # Validation
        # ==================================================

        if total_score is None:

            print(
                f"[WARNING] {symbol}: "
                "Total score is None."
            )

            return

        if trend_result is None:

            trend_result = {}

        if sr_result is None:

            sr_result = {}

        if pattern_result is None:

            pattern_result = {}

        if latest_row is None:

            print(
                f"[WARNING] {symbol}: "
                "Latest row is None."
            )

            return

        # ==================================================
        # Final Score
        # ==================================================

        signal = total_score.get(
            "Signal",
            "HOLD"
        )

        score = self._safe_value(
            total_score.get(
                "Score",
                0
            ),
            0
        )

        confidence = self._safe_value(
            total_score.get(
                "Confidence",
                0
            ),
            0
        )

        bullish = total_score.get(
            "Bullish",
            False
        )

        bearish = total_score.get(
            "Bearish",
            False
        )

        # ==================================================
        # Support / Resistance
        # ==================================================

        support = self._extract_support(
            sr_result
        )

        resistance = self._extract_resistance(
            sr_result
        )

        support_price = self._safe_value(
            support.get("price")
        )

        resistance_price = self._safe_value(
            resistance.get("price")
        )

        # ==================================================
        # Pattern Information
        # ==================================================

        best_pattern = pattern_result.get(
            "Pattern"
        )

        pattern_score = self._safe_value(
            pattern_result.get(
                "Pattern Score",
                0
            ),
            0
        )

        pattern_confidence = self._safe_value(
            pattern_result.get(
                "Confidence",
                0
            ),
            0
        )

        detected_patterns = pattern_result.get(
            "Detected Patterns",
            []
        )

        bullish_patterns = pattern_result.get(
            "Bullish Patterns",
            []
        )

        bearish_patterns = pattern_result.get(
            "Bearish Patterns",
            []
        )

        neutral_patterns = pattern_result.get(
            "Neutral Patterns",
            []
        )

        pattern_bias = pattern_result.get(
            "Market Bias"
        )

        # ==================================================
        # Score Engine Details
        # ==================================================

        score_details = total_score.get(
            "Details",
            {}
        )

        if not isinstance(
            score_details,
            dict
        ):

            score_details = {}

        # ==================================================
        # Module Scores
        # ==================================================

        indicator_score = self._get_module_score(
            total_score,
            "Indicator Score",
            "Indicator Score"
        )

        trend_score = self._get_module_score(
            total_score,
            "Trend Score",
            "Trend Score"
        )

        sr_score = self._get_module_score(
            total_score,
            "Support/Resistance Score",
            "Support Resistance Score"
        )

        pattern_score_module = self._get_module_score(
            total_score,
            "Pattern Score",
            "Pattern Score"
        )

        # ==================================================
        # Raw Scores
        # ==================================================

        indicator_raw_score = self._safe_value(
            indicator_score.get(
                "Score",
                0
            ),
            0
        )

        trend_raw_score = self._safe_value(
            trend_score.get(
                "Score",
                0
            ),
            0
        )

        sr_raw_score = self._safe_value(
            sr_score.get(
                "Score",
                0
            ),
            0
        )

        pattern_raw_score = self._safe_value(
            pattern_score_module.get(
                "Score",
                0
            ),
            0
        )

        # ==================================================
        # Weighted Scores
        # ==================================================

        weighted_indicator = self._safe_value(
            score_details.get(
                "Weighted Indicator",
                0
            ),
            0
        )

        weighted_trend = self._safe_value(
            score_details.get(
                "Weighted Trend",
                0
            ),
            0
        )

        weighted_support = self._safe_value(
            score_details.get(
                "Weighted Support",
                0
            ),
            0
        )

        weighted_pattern = self._safe_value(
            score_details.get(
                "Weighted Pattern",
                0
            ),
            0
        )

        # ==================================================
        # Votes
        # ==================================================

        bullish_votes = score_details.get(
            "Bullish Votes",
            0
        )

        bearish_votes = score_details.get(
            "Bearish Votes",
            0
        )

        # ==================================================
        # Conflicts
        # ==================================================

        directional_conflict = score_details.get(
            "Directional Conflict",
            False
        )

        pattern_conflict = False

        pattern_module_details = (
            pattern_score_module.get(
                "Details",
                {}
            )
        )

        if isinstance(
            pattern_module_details,
            dict
        ):

            pattern_conflict = pattern_module_details.get(
                "Conflict",
                False
            )

        # ==================================================
        # Latest Price
        # ==================================================

        close = self._safe_value(
            latest_row.get("Close")
        )

        # ==================================================
        # Date / Time
        # ==================================================

        date = self._safe_value(
            latest_row.get("Date")
        )

        time = self._safe_value(
            latest_row.get("Time")
        )

        datetime_value = self._safe_value(
            latest_row.get("DateTime")
        )

        # ==================================================
        # Technical Indicators
        # ==================================================

        ema20 = self._safe_value(
            latest_row.get("EMA20")
        )

        ema50 = self._safe_value(
            latest_row.get("EMA50")
        )

        ema200 = self._safe_value(
            latest_row.get("EMA200")
        )

        rsi = self._safe_value(
            latest_row.get("RSI")
        )

        macd = self._safe_value(
            latest_row.get("MACD")
        )

        macd_signal = self._safe_value(
            latest_row.get("MACD_SIGNAL")
        )

        macd_hist = self._safe_value(
            latest_row.get("MACD_HIST")
        )

        adx = self._safe_value(
            latest_row.get("ADX")
        )

        atr = self._safe_value(
            latest_row.get("ATR")
        )

        atr_ma = self._safe_value(
            latest_row.get("ATR_MA")
        )

        rvol = self._safe_value(
            latest_row.get("RVOL")
        )

        volume = self._safe_value(
            latest_row.get("Volume")
        )

        volume_ma = self._safe_value(
            latest_row.get("VOLUME_MA")
        )

        # ==================================================
        # Trend Information
        # ==================================================

        trend = trend_result.get(
            "Trend"
        )

        trend_signal = trend_result.get(
            "Signal"
        )

        trend_confidence = self._safe_value(
            trend_result.get(
                "Confidence",
                0
            ),
            0
        )

        # ==================================================
        # Final Reasons
        # ==================================================

        final_reasons = total_score.get(
            "Reasons",
            []
        )

        # ==================================================
        # Complete Analysis Row
        # ==================================================

        row = {

            # ----------------------------------------------
            # Basic
            # ----------------------------------------------

            "Symbol":
                symbol,

            "Date":
                date,

            "Time":
                time,

            "DateTime":
                datetime_value,

            "Close":
                close,

            # ----------------------------------------------
            # Final Decision
            # ----------------------------------------------

            "Signal":
                signal,

            "Final Score":
                score,

            "Confidence":
                confidence,

            "Bullish":
                bullish,

            "Bearish":
                bearish,

            # ----------------------------------------------
            # Trend
            # ----------------------------------------------

            "Trend":
                trend,

            "Trend Signal":
                trend_signal,

            "Trend Confidence":
                trend_confidence,

            # ----------------------------------------------
            # Price Action Pattern
            # ----------------------------------------------

            "Best Pattern":
                best_pattern,

            "Pattern Score":
                pattern_score,

            "Pattern Confidence":
                pattern_confidence,

            "Pattern Bias":
                pattern_bias,

            "Detected Patterns":
                self._list_to_text(
                    detected_patterns
                ),

            "Bullish Patterns":
                self._list_to_text(
                    bullish_patterns
                ),

            "Bearish Patterns":
                self._list_to_text(
                    bearish_patterns
                ),

            "Neutral Patterns":
                self._list_to_text(
                    neutral_patterns
                ),

            "Pattern Conflict":
                pattern_conflict,

            # ----------------------------------------------
            # Raw Module Scores
            # ----------------------------------------------

            "Indicator Score":
                indicator_raw_score,

            "Trend Score":
                trend_raw_score,

            "Support/Resistance Score":
                sr_raw_score,

            "Price Action Score":
                pattern_raw_score,

            # ----------------------------------------------
            # Weighted Scores
            # ----------------------------------------------

            "Weighted Indicator":
                weighted_indicator,

            "Weighted Trend":
                weighted_trend,

            "Weighted Support":
                weighted_support,

            "Weighted Pattern":
                weighted_pattern,

            # ----------------------------------------------
            # Votes
            # ----------------------------------------------

            "Bullish Votes":
                bullish_votes,

            "Bearish Votes":
                bearish_votes,

            "Directional Conflict":
                directional_conflict,

            # ----------------------------------------------
            # Support / Resistance
            # ----------------------------------------------

            "Major Support":
                support_price,

            "Major Resistance":
                resistance_price,

            # ----------------------------------------------
            # Indicators
            # ----------------------------------------------

            "EMA20":
                ema20,

            "EMA50":
                ema50,

            "EMA200":
                ema200,

            "RSI":
                rsi,

            "MACD":
                macd,

            "MACD Signal":
                macd_signal,

            "MACD Histogram":
                macd_hist,

            "ADX":
                adx,

            "ATR":
                atr,

            "ATR MA":
                atr_ma,

            "RVOL":
                rvol,

            "Volume":
                volume,

            "Volume MA":
                volume_ma,

            # ----------------------------------------------
            # Reasons
            # ----------------------------------------------

            "Reasons":
                self._list_to_text(
                    final_reasons
                )
        }

        # ==================================================
        # 1. ALWAYS UPDATE DAILY_ANALYSIS.XLSX
        # ==================================================

        daily_filename = os.path.join(
            self.output_folder,
            "Daily_Analysis.xlsx"
        )

        self._upsert_row(
            daily_filename,
            row
        )

        # ==================================================
        # 2. REMOVE SYMBOL FROM ALL CATEGORY FILES
        # ==================================================
        #
        # Important:
        #
        # If yesterday:
        #     KALYANKJIL = BUY
        #
        # and today:
        #     KALYANKJIL = HOLD
        #
        # it must not remain inside
        # Bullish_Shares.xlsx.
        #
        # ==================================================

        category_files = [

            os.path.join(
                self.output_folder,
                "Bullish_Shares.xlsx"
            ),

            os.path.join(
                self.output_folder,
                "Bearish_Shares.xlsx"
            ),

            os.path.join(
                self.output_folder,
                "Hold_Shares.xlsx"
            )

        ]

        for filename in category_files:

            self._remove_symbol_from_file(
                filename,
                symbol
            )

        # ==================================================
        # 3. CATEGORY REPORT
        # ==================================================
        #
        # Daily_Analysis:
        #     ALL shares
        #
        # Category reports:
        #     Only shares meeting confidence filter
        #
        # ==================================================

        if confidence >= self.minimum_confidence:

            category_filename = (
                self._get_category_filename(
                    signal
                )
            )

            self._upsert_row(
                category_filename,
                row
            )

            print(
                f"{symbol} saved | "
                f"{signal} | "
                f"Score={score} | "
                f"Confidence={confidence}"
            )

        else:

            print(
                f"{symbol} saved to "
                f"Daily_Analysis.xlsx | "
                f"Category skipped | "
                f"Confidence={confidence} < "
                f"{self.minimum_confidence}"
            )

    # ======================================================
    # Save All Shares
    # ======================================================

    def save_all(
        self,
        analysis_results
    ):
        """
        Optional helper for saving multiple shares.

        analysis_results should be a list of dictionaries:

        [
            {
                "Symbol": ...,
                "Total Score": ...,
                "Trend": ...,
                "SR": ...,
                "Pattern": ...,
                "Latest Row": ...
            }
        ]

        This method is optional.
        """

        if analysis_results is None:

            return

        for result in analysis_results:

            try:

                self.save_share(

                    symbol=result["Symbol"],

                    total_score=result["Total Score"],

                    trend_result=result.get(
                        "Trend",
                        {}
                    ),

                    sr_result=result.get(
                        "SR",
                        {}
                    ),

                    pattern_result=result.get(
                        "Pattern",
                        {}
                    ),

                    latest_row=result["Latest Row"]

                )

            except Exception as e:

                symbol = result.get(
                    "Symbol",
                    "UNKNOWN"
                )

                print(
                    f"[ERROR] {symbol}: "
                    f"Report generation failed: {e}"
                )

In [39]:
# ==========================================================
# DAILY ANALYSIS - ALL SYMBOLS
# ==========================================================

import traceback

from IndicatorEngine import IndicatorEngine
from PriceActionPatternEngine import PriceActionPatternEngine
from TrendEngine import TrendEngine
from SupportResistanceEngine import SupportResistanceEngine
from ScoreEngine import ScoreEngine


# ==========================================================
# Configuration
# ==========================================================

SYMBOLS_FILE = "/home/hadoop/shares.txt"

DATA_FOLDER = "/home/hadoop/shareMarket_Data/weekly/"

REPORT_FOLDER = "report/weekly"

MINIMUM_CONFIDENCE = 60


# ==========================================================
# Main
# ==========================================================

def main():

    print("\n" + "=" * 80)
    print("DAILY STOCK ANALYSIS - ALL SHARES")
    print("=" * 80)

    # ======================================================
    # 1. Data Loader
    # ======================================================

    data_loader = DataLoader(
        symbols_file=SYMBOLS_FILE,
        data_folder=DATA_FOLDER
    )

    all_daily_data = data_loader.load_all_daily()

    print(
        f"\nSuccessfully Loaded : "
        f"{len(all_daily_data)} shares"
    )

    # ======================================================
    # 2. Initialize Engines
    # ======================================================

    print("\nInitializing engines...")

    indicator_engine = IndicatorEngine()

    pattern_engine = PriceActionPatternEngine()

    trend_engine = TrendEngine()

    sr_engine = SupportResistanceEngine()

    score_engine = ScoreEngine()

    report_writer = ReportWriter(
        output_folder=REPORT_FOLDER,
        minimum_confidence=MINIMUM_CONFIDENCE
    )

    print("\nAll engines initialized.")

    # ======================================================
    # Counters
    # ======================================================

    processed = 0
    failed = 0

    buy_count = 0
    sell_count = 0
    hold_count = 0

    # ======================================================
    # Process All Shares
    # ======================================================

    for index, symbol in enumerate(
        all_daily_data.keys(),
        start=1
    ):

        print("\n" + "=" * 80)

        print(
            f"[{index}/{len(all_daily_data)}] "
            f"PROCESSING : {symbol}"
        )

        print("=" * 80)

        try:

            # ==================================================
            # Get Daily Data
            # ==================================================

            daily_df = all_daily_data[symbol].copy()

            if daily_df.empty:

                raise ValueError(
                    "Daily dataframe is empty"
                )

            print(
                "Rows         :",
                len(daily_df)
            )

            print(
                "Latest Close :",
                daily_df.iloc[-1]["Close"]
            )

            # ==================================================
            # Indicator Engine
            # ==================================================

            daily_df = indicator_engine.calculate(
                daily_df
            )

            # ==================================================
            # Price Action Pattern
            # ==================================================

            pattern_result = pattern_engine.detect(
                daily_df
            )

            print(
                "Pattern      :",
                pattern_result.get("Pattern")
            )

            print(
                "Pattern Score:",
                pattern_result.get("Pattern Score")
            )

            # ==================================================
            # Trend
            # ==================================================

            trend_result = trend_engine.detect(
                daily_df
            )

            print(
                "Trend        :",
                trend_result.get("Trend")
            )

            print(
                "Trend Score  :",
                trend_result.get("Score")
            )

            # ==================================================
            # Support / Resistance
            # ==================================================

            sr_result = sr_engine.calculate(
                daily_df
            )

            # ==================================================
            # Final Score
            # ==================================================

            final_result = score_engine.calculate(

                df=daily_df,

                trend_result=trend_result,

                sr_result=sr_result,

                pattern_result=pattern_result

            )

            signal = final_result.get(
                "Signal",
                "HOLD"
            )

            score = final_result.get(
                "Score",
                0
            )

            confidence = final_result.get(
                "Confidence",
                0
            )

            # ==================================================
            # Print Result
            # ==================================================

            print("\n" + "-" * 80)

            print("FINAL RESULT")

            print("-" * 80)

            print(
                "Symbol     :",
                symbol
            )

            print(
                "Signal     :",
                signal
            )

            print(
                "Score      :",
                score
            )

            print(
                "Confidence :",
                confidence
            )

            print(
                "Bullish    :",
                final_result.get("Bullish")
            )

            print(
                "Bearish    :",
                final_result.get("Bearish")
            )

            # ==================================================
            # Save to Daily_Analysis.xlsx
            # ==================================================

            latest_row = daily_df.iloc[-1]

            report_writer.save_share(

                symbol=symbol,

                total_score=final_result,

                trend_result=trend_result,

                sr_result=sr_result,

                pattern_result=pattern_result,

                latest_row=latest_row

            )

            # ==================================================
            # Counters
            # ==================================================

            processed += 1

            if signal == "BUY":

                buy_count += 1

            elif signal == "SELL":

                sell_count += 1

            else:

                hold_count += 1

        except Exception as e:

            failed += 1

            print("\n" + "!" * 80)

            print(
                f"[FAILED] {symbol}"
            )

            print(
                "Error:",
                str(e)
            )

            print("!" * 80)

            traceback.print_exc()

            continue

    # ==========================================================
    # FINAL SUMMARY
    # ==========================================================

    print("\n" + "=" * 80)

    print("DAILY ANALYSIS COMPLETED")

    print("=" * 80)

    print(
        "Total Loaded :",
        len(all_daily_data)
    )

    print(
        "Processed     :",
        processed
    )

    print(
        "Failed        :",
        failed
    )

    print(
        "BUY           :",
        buy_count
    )

    print(
        "SELL          :",
        sell_count
    )

    print(
        "HOLD          :",
        hold_count
    )

    print("\nReport:")

    print(
        "  report/weekly/Weekly_Analysis.xlsx"
    )

    print("=" * 80)


# ==========================================================
# Entry Point
# ==========================================================

if __name__ == "__main__":
    main()


DAILY STOCK ANALYSIS - ALL SHARES
Total Shares : 80

Successfully Loaded : 80 shares

Initializing engines...
Indicator Engine Initialized
Price Action Pattern Engine initialize
Trend Engine Initialized
Support & Resistance Engine Initialized
Score Engine V2 Initialized
Report Writer V3 Initialized
Output Folder : report/weekly

All engines initialized.

[1/80] PROCESSING : ADANIPORTS
Rows         : 239
Latest Close : 1693.5
Pattern      : Rounding Bottom
Pattern Score: 65.0
Trend        : Neutral
Trend Score  : -20

--------------------------------------------------------------------------------
FINAL RESULT
--------------------------------------------------------------------------------
Symbol     : ADANIPORTS
Signal     : HOLD
Score      : 3.53
Confidence : 51.85
Bullish    : False
Bearish    : False
ADANIPORTS saved to Daily_Analysis.xlsx | Category skipped | Confidence=51.85 < 60

[2/80] PROCESSING : ADANIPOWER
Rows         : 239
Latest Close : 208.25
[WARNING] Rounding Bottom de

Traceback (most recent call last):
  File "/tmp/ipykernel_11465/208074452.py", line 133, in main
    daily_df = indicator_engine.calculate(
        daily_df
    )
  File "/home/devinderjeet/IndicatorEngine.py", line 491, in calculate
    self.validate(df)
    ~~~~~~~~~~~~~^^^^
  File "/home/devinderjeet/IndicatorEngine.py", line 480, in validate
    raise ValueError(
        "Need minimum 200 candles."
    )
ValueError: Need minimum 200 candles.


Pattern      : Breakout
Pattern Score: 25.0
Trend        : Bullish
Trend Score  : 73

--------------------------------------------------------------------------------
FINAL RESULT
--------------------------------------------------------------------------------
Symbol     : AEGISLOG
Signal     : HOLD
Score      : 46.35
Confidence : 72.98
Bullish    : False
Bearish    : False
AEGISLOG saved | HOLD | Score=46.35 | Confidence=72.98

[61/80] PROCESSING : AFFLE
Rows         : 239
Latest Close : 1645.6
Pattern      : Falling Wedge
Pattern Score: 25.0
Trend        : Neutral
Trend Score  : 23

--------------------------------------------------------------------------------
FINAL RESULT
--------------------------------------------------------------------------------
Symbol     : AFFLE
Signal     : HOLD
Score      : 20.55
Confidence : 56.23
Bullish    : False
Bearish    : False
AFFLE saved to Daily_Analysis.xlsx | Category skipped | Confidence=56.23 < 60

[62/80] PROCESSING : IDBI
Rows         : 